# DX-COM Tutorial 2: Intermediate

This notebook moves from a successful compile to a controlled and explainable compilation workflow.

You will improve calibration decisions, build real Model Zoo models with PPU Type 0 and Type 1, and measure the effect of TopK-first optimization on YOLO26n.

## Learning objectives

By the end, you should be able to:

- select representative calibration data and reproduce model preprocessing,
- explain which detection operations run on the NPU PPU and which remain on the host CPU,
- choose PPU Type 0 or Type 1 from the detection-head architecture,
- validate PPU layer mappings against an ONNX graph,
- compile Model Zoo YOLOv7 and YOLOX-S models with hardware PPU,
- compare PPU and non-PPU Model Zoo DXNN runtime behavior,
- apply TopK-first optimization to YOLO26n, and
- compare baseline and optimized DXNN performance with `dxrun`.

## 1. Initialize the tutorial workspace

In [ ]:
from pathlib import Path
import json
import os
import re
import shlex
import shutil
import subprocess
import sys

# Find the dx-tutorials root regardless of where Jupyter was started.
TUTORIAL_ROOT = Path.cwd().resolve()
while TUTORIAL_ROOT != TUTORIAL_ROOT.parent:
    if (TUTORIAL_ROOT / "tutorial_paths.py").is_file():
        break
    TUTORIAL_ROOT = TUTORIAL_ROOT.parent
else:
    raise FileNotFoundError("tutorial_paths.py was not found. Start Jupyter from the dx-tutorials repository.")

sys.path.insert(0, str(TUTORIAL_ROOT))
from tutorial_paths import DX_ALL_SUITE_DIR, DX_COMPILER_DIR

DX_COM_DIR = DX_COMPILER_DIR / "dx_com"
DX_COMPILER_VENV = DX_COMPILER_DIR / "venv-dx-compiler-local"
DXCOM_PATH = DX_COMPILER_VENV / "bin" / "dxcom"
DXCOM_PYTHON = DX_COMPILER_VENV / "bin" / "python"
DXPARSE_PATH = shutil.which("dxparse")
DXRUN_PATH = shutil.which("dxrun")

for required_path in (DX_COM_DIR, DXCOM_PATH, DXCOM_PYTHON):
    if not required_path.exists():
        raise FileNotFoundError(f"Required DX-COM path does not exist: {required_path}")
for command_name, command_path in (("dxparse", DXPARSE_PATH), ("dxrun", DXRUN_PATH)):
    if not command_path:
        raise FileNotFoundError(f"{command_name} was not found in PATH. Complete the SDK installation first.")

WORK_DIR = TUTORIAL_ROOT / "notebooks/T05-DX-Compiler"
MODEL_DIR = WORK_DIR / "models"
CONFIG_DIR = WORK_DIR / "configs"
OUTPUT_DIR = WORK_DIR / "outputs"

for path in (WORK_DIR, MODEL_DIR, CONFIG_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

# Reuse the SDK sample images without copying them into this workspace.
CALIBRATION_SOURCE = DX_COM_DIR / "calibration_dataset"
CALIBRATION_DIR = WORK_DIR / "calibration_dataset"
if not CALIBRATION_SOURCE.is_dir():
    raise FileNotFoundError(f"Calibration dataset was not found: {CALIBRATION_SOURCE}")
if CALIBRATION_DIR.is_symlink() and not CALIBRATION_DIR.exists():
    CALIBRATION_DIR.unlink()
if not CALIBRATION_DIR.exists():
    CALIBRATION_DIR.symlink_to(CALIBRATION_SOURCE, target_is_directory=True)
if CALIBRATION_DIR.resolve() != CALIBRATION_SOURCE.resolve():
    raise RuntimeError(f"Calibration path points to an unexpected location: {CALIBRATION_DIR}")

os.chdir(WORK_DIR)
print(f"DX-COM       : {DXCOM_PATH}")
print(f"Workspace    : {WORK_DIR}")
print(f"Calibration  : {CALIBRATION_DIR} -> {CALIBRATION_SOURCE}")

In [ ]:
try:
    import onnx
except ImportError:
    subprocess.run(
        ["uv", "pip", "install", "--python", sys.executable, "onnx"],
        check=True,
    )
    import onnx

print("ONNX:", onnx.__version__)

## 2. Calibration is part of the model definition

Quantization observes floating-point tensors produced by the calibration inputs. A convenient but unrelated image set can produce a DXNN file while degrading accuracy.

| Decision | Recommended practice |
|---|---|
| Samples | Cover real lighting, scale, background, and class distributions |
| Preprocessing | Match training and evaluation exactly |
| Count | Start with the model recipe; increase only after measuring stability |
| Method | Start with the published configuration, then compare methods using metrics |
| Validation | Compare ONNX and DXNN outputs on a held-out dataset |

The SDK `calibration_dataset` link keeps this tutorial runnable, but it is not a substitute for a task-specific dataset.

### 2.1 Preprocessing order

A typical detection pipeline is:

```text
image → letterbox/pad → BGR-to-RGB → divide by 255 → HWC-to-CHW → add batch axis
```

Do not copy this sequence blindly. Inspect the model exporter and training code. A different pad location, pad value, color order, or normalization changes the calibration tensor distribution.

## 3. PPU overview

The **Post-Processing Unit (PPU)** is hardware inside the DEEPX NPU. For supported YOLO detection heads, it reduces the number of raw candidates that the host CPU must handle.

The PPU performs two selected operations:

1. **confidence filtering** removes candidates below the compile-time threshold, and
2. **class prediction** selects the strongest class for each remaining candidate.

The PPU does **not** perform Non-Maximum Suppression (NMS). NMS and application-level interpretation still run on the host CPU. The PPU also does not replace image preprocessing or NPU inference.

### 3.1 Where PPU fits in the detection pipeline

```text
Without hardware PPU
┌──────────┐   ┌───────────────┐   ┌──────────────────────────────┐   ┌─────────┐
│ Input    │ → │ NPU inference │ → │ CPU filtering/class selection│ → │ CPU NMS │
└──────────┘   └───────────────┘   └──────────────────────────────┘   └─────────┘

With PPU Type 0 or Type 1
┌──────────┐   ┌───────────────┐   ┌────────────────────────────┐   ┌─────────┐
│ Input    │ → │ NPU inference │ → │ PPU filter/class prediction│ → │ CPU NMS │
└──────────┘   └───────────────┘   └────────────────────────────┘   └─────────┘
                                           hardware                    host
```

| Stage | Without PPU | With PPU Type 0/1 |
|---|---|---|
| Neural-network inference | NPU | NPU |
| Confidence filtering | Host CPU | PPU hardware |
| Best-class selection | Host CPU | PPU hardware |
| Bounding-box decode | Model/type dependent | Model/type dependent |
| NMS | Host CPU | Host CPU |

The main benefit is lower host-CPU work and less intermediate detection data. The exact end-to-end gain depends on the model, threshold, scene, host CPU, and application post-processing.

### 3.2 Choose the mode from the model architecture

| Mode | Architecture | Typical models | Layer mapping | Execution |
|---|---|---|---|---|
| PPU Type 0 | Anchor-based | YOLOv3, YOLOv4, YOLOv5, YOLOv7 | Detection Conv node → number of anchors | PPU hardware |
| PPU Type 1 | Anchor-free | YOLOX, YOLOv8–YOLOv12 | `bbox`, `obj_conf` when present, and `cls_conf` nodes | PPU hardware |
| `pre_optimize()` | TopK-first ONNX rewrite | YOLOv8-family, YOLOv10, YOLO26 | Per-scale bbox/class output tensors | NPU + host CPU graph |

Do not select a type from the model name alone. Confirm the exported ONNX head structure and use node names from that exact file.

### 3.3 Read a PPU configuration

```text
PPU configuration
├── type           selects the supported head architecture
├── conf_thres     fixed confidence threshold compiled into the DXNN
├── num_classes    class count of this exported model
├── activation     Type 0 activation, usually Sigmoid
└── layer          exact ONNX head-node mapping
```

| Field | Type 0 | Type 1 | Why it matters |
|---|:---:|:---:|---|
| `type` | Required | Required | Selects the hardware data path |
| `conf_thres` | Required | Required | Controls how many candidates leave the PPU |
| `num_classes` | Required | Required | Must match head-channel layout |
| `activation` | Required | Not used | Applies the anchor-based score activation |
| `layer` | Dictionary | List | Connects PPU inputs to exact ONNX nodes |
| `num_anchors` | Per layer | Not used | Must match the anchor count at each scale |
| `obj_conf` | Not used | Model dependent | YOLOX has a separate objectness branch |

**Important:** `conf_thres` is fixed during compilation. Changing it later requires a new DXNN. A higher value reduces host work but can also remove valid detections. Measure accuracy before deployment.

## 4. PPU Type 0 lab: YOLOv7 from the Model Zoo

YOLOv7 uses an anchor-based detection head, so this lab uses PPU Type 0. You will download the actual Model Zoo ONNX and its matching PPU JSON, verify the mapped Conv nodes, adapt the dataset path, and compile a DXNN.

### 4.1 Download the Model Zoo files safely

If a non-empty destination file already exists, the helper skips the download without making a network request. Delete that local file first when you need to download a fresh copy. For a new download, the helper checks the remote file size, writes to a temporary `.part` file, and replaces the destination only after a complete transfer. A DNS or network failure raises an exception and does not leave an empty final file.

This lab downloads the ONNX and PPU JSON for compilation. It also downloads the published non-PPU DXNN as the benchmark reference.

In [ ]:
from urllib.request import Request, urlopen

def download_file(url, destination, chunk_size=1024 * 1024):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.is_file() and destination.stat().st_size > 0:
        print(f"Skip download: found {destination} ({destination.stat().st_size:,} bytes)")
        return destination

    if destination.is_file():
        print(f"Existing file is empty; downloading again: {destination}")

    with urlopen(Request(url, method="HEAD"), timeout=30) as response:
        expected_size = int(response.headers.get("Content-Length") or 0)

    temporary = destination.with_name(destination.name + ".part")
    temporary.unlink(missing_ok=True)
    downloaded = 0
    next_report = 10

    try:
        with urlopen(url, timeout=60) as response, temporary.open("wb") as output:
            while True:
                chunk = response.read(chunk_size)
                if not chunk:
                    break
                output.write(chunk)
                downloaded += len(chunk)
                if expected_size:
                    percent = downloaded * 100 // expected_size
                    if percent >= next_report:
                        print(f"{destination.name}: {min(percent, 100)}%")
                        next_report += 10

        if expected_size and downloaded != expected_size:
            raise IOError(f"Incomplete download: expected {expected_size} bytes, received {downloaded}")
        temporary.replace(destination)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise

    print(f"Saved: {destination} ({downloaded:,} bytes)")
    return destination

def tensor_shape(value_info):
    return [dim.dim_value or dim.dim_param for dim in value_info.type.tensor_type.shape.dim]

In [ ]:
YOLOV7_ONNX_URL = "https://sdk.deepx.ai/modelzoo/onnx/yolov7_640x640.onnx"
YOLOV7_PPU_JSON_URL = "https://sdk.deepx.ai/modelzoo/q-lite-json/2_4_0/yolov7_640x640_ppu.json"
YOLOV7_NON_PPU_DXNN_URL = "https://sdk.deepx.ai/modelzoo/dxnn/2_4_0/yolov7_640x640.dxnn"
YOLOV7_ONNX = MODEL_DIR / "yolov7_640x640.onnx"
YOLOV7_PPU_JSON = CONFIG_DIR / "yolov7_640x640_ppu.modelzoo.json"
YOLOV7_NON_PPU_DXNN = MODEL_DIR / "yolov7_640x640_non_ppu.dxnn"

download_file(YOLOV7_ONNX_URL, YOLOV7_ONNX)
download_file(YOLOV7_PPU_JSON_URL, YOLOV7_PPU_JSON)
download_file(YOLOV7_NON_PPU_DXNN_URL, YOLOV7_NON_PPU_DXNN)

### 4.2 Inspect the ONNX and Type 0 mapping

For Type 0, `layer` is a dictionary. Each key must be the name of a detection-head Conv node, and `num_anchors` must match that scale. The output channel count follows:

```text
channels = num_anchors × (5 + num_classes)
         = 3 × (5 + 80)
         = 255
```

In [ ]:
yolov7_model = onnx.load(YOLOV7_ONNX)
onnx.checker.check_model(yolov7_model)
yolov7_config = json.loads(YOLOV7_PPU_JSON.read_text())

print("IR/opset:", yolov7_model.ir_version, [(item.domain or "ai.onnx", item.version) for item in yolov7_model.opset_import])
print("Inputs:", [(value.name, tensor_shape(value)) for value in yolov7_model.graph.input])
print("Outputs:", [(value.name, tensor_shape(value)) for value in yolov7_model.graph.output])
print(json.dumps(yolov7_config["ppu"], indent=2))

node_names = {node.name for node in yolov7_model.graph.node}
type0_layers = yolov7_config["ppu"]["layer"]
missing_type0_nodes = sorted(set(type0_layers) - node_names)
if missing_type0_nodes:
    raise ValueError(f"Type 0 nodes were not found in this ONNX: {missing_type0_nodes}")
print("Verified Type 0 nodes:", list(type0_layers))

The highlighted Conv nodes are the three scale-specific detection heads used by the Model Zoo PPU configuration.

![YOLOv7 Type 0 PPU head mapping](assets/yolov7-class-n80-ppu.png)

### 4.3 Adapt only the environment-specific path

The Model Zoo JSON contains the preprocessing recipe used for this model. Keep it unchanged for the lab and replace only the unavailable calibration-dataset path. For a product model, use a representative dataset from the deployment domain.

In [ ]:
yolov7_local_config = json.loads(YOLOV7_PPU_JSON.read_text())
yolov7_local_config["default_loader"]["dataset_path"] = "./calibration_dataset"
YOLOV7_LOCAL_JSON = CONFIG_DIR / "yolov7_640x640_ppu.local.json"
YOLOV7_LOCAL_JSON.write_text(json.dumps(yolov7_local_config, indent=2) + "\n")

print("Local config:", YOLOV7_LOCAL_JSON)
print("Dataset     :", yolov7_local_config["default_loader"]["dataset_path"])
print("PPU type    :", yolov7_local_config["ppu"]["type"])

### 4.4 Compile and inspect the Type 0 DXNN

The next code cell activates the DX-COM environment and runs `dxcom` directly. It is equivalent to entering the following commands in a separate terminal:

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/T05-DX-Compiler
dxcom -m models/yolov7_640x640.onnx \
      -c configs/yolov7_640x640_ppu.local.json \
      -o outputs/yolov7_type0_ppu \
      --gen_log \
      --export_html
```

The actual SDK and tutorial paths may differ. The code cell uses the paths loaded from `config.json`. If the output directory already contains a DXNN file, it prints a skip message and does not run `dxcom` again.

In [ ]:
YOLOV7_PPU_OUTPUT = OUTPUT_DIR / "yolov7_type0_ppu"

yolov7_dxnn_files = list(YOLOV7_PPU_OUTPUT.glob("*.dxnn"))
if yolov7_dxnn_files:
    print(f"Skip compilation: found {yolov7_dxnn_files[0]}")
else:
    !source "{DX_COMPILER_VENV}/bin/activate" && \
      dxcom -m "{YOLOV7_ONNX}" \
            -c "{YOLOV7_LOCAL_JSON}" \
            -o "{YOLOV7_PPU_OUTPUT}" \
            --gen_log \
            --export_html

yolov7_dxnn_files = list(YOLOV7_PPU_OUTPUT.glob("*.dxnn"))
if not yolov7_dxnn_files:
    raise FileNotFoundError("Type 0 compilation did not produce a DXNN file.")
YOLOV7_PPU_DXNN = max(yolov7_dxnn_files, key=lambda path: path.stat().st_mtime_ns)
print("DXNN:", YOLOV7_PPU_DXNN)

!dxparse -m "{YOLOV7_PPU_DXNN}" -v

### 4.5 Compare PPU and non-PPU with `dxrun`

The non-PPU DXNN is downloaded directly from the Model Zoo, while the PPU DXNN is the result compiled in Section 4.4. The next cell runs both models for five seconds with synthetic input:

```bash
dxrun -m models/yolov7_640x640_non_ppu.dxnn --use-ort -t 5
dxrun -m outputs/yolov7_type0_ppu/<compiled-model>.dxnn --use-ort -t 5
```

The non-PPU model contains NPU and CPU tasks and returns raw detection values. The PPU model returns hardware-generated `BBOX` data and reduces the output handled by the host. PPU does **not** guarantee higher synthetic-input FPS: model scheduling, output transfer, and device state can make the non-PPU result faster in this isolated test. For a product decision, also measure full-pipeline latency and host CPU usage with real input.

In [ ]:
from IPython.display import Markdown, display

def benchmark_model(label, model_path, seconds=5):
    command = [str(DXRUN_PATH), "-m", str(model_path), "--use-ort", "-t", str(seconds)]
    print(f"\n===== {label} =====")
    print("$", shlex.join(command))
    result = subprocess.run(
        command,
        cwd=WORK_DIR,
        check=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    match = re.search(r"FPS\s*:\s*([0-9.]+)", result.stdout)
    if not match:
        raise RuntimeError(f"Could not read FPS from dxrun output for {label}.")
    return float(match.group(1))

def compare_ppu_runtime(non_ppu_model, ppu_model, ppu_label):
    non_ppu_fps = benchmark_model("Non-PPU Model Zoo DXNN", non_ppu_model)
    ppu_fps = benchmark_model(ppu_label, ppu_model)
    ratio = ppu_fps / non_ppu_fps
    table = (
        "| Variant | FPS | Relative to non-PPU |\n"
        "|---|---:|---:|\n"
        f"| Non-PPU Model Zoo DXNN | {non_ppu_fps:.2f} | 1.00x |\n"
        f"| {ppu_label} | {ppu_fps:.2f} | {ratio:.2f}x |"
    )
    display(Markdown(table))
    return {"non_ppu_fps": non_ppu_fps, "ppu_fps": ppu_fps, "ratio": ratio}

yolov7_ppu_comparison = compare_ppu_runtime(
    YOLOV7_NON_PPU_DXNN,
    YOLOV7_PPU_DXNN,
    "Type 0 PPU DXNN",
)

## 5. PPU Type 1 lab: YOLOX-S from the Model Zoo

YOLOX uses a decoupled anchor-free detection head. Each scale has separate bounding-box, objectness, and class-confidence branches, so this lab uses PPU Type 1.

### 5.1 Download the ONNX and Type 1 JSON

The same safe downloader is reused. The ONNX and PPU JSON are used for compilation, while the published non-PPU DXNN is used as the benchmark reference.

In [ ]:
YOLOX_ONNX_URL = "https://sdk.deepx.ai/modelzoo/onnx/yolox-s_640x640.onnx"
YOLOX_PPU_JSON_URL = "https://sdk.deepx.ai/modelzoo/q-lite-json/2_4_0/yolox-s_640x640_ppu.json"
YOLOX_NON_PPU_DXNN_URL = "https://sdk.deepx.ai/modelzoo/dxnn/2_4_0/yolox-s_640x640.dxnn"
YOLOX_ONNX = MODEL_DIR / "yolox-s_640x640.onnx"
YOLOX_PPU_JSON = CONFIG_DIR / "yolox-s_640x640_ppu.modelzoo.json"
YOLOX_NON_PPU_DXNN = MODEL_DIR / "yolox-s_640x640_non_ppu.dxnn"

download_file(YOLOX_ONNX_URL, YOLOX_ONNX)
download_file(YOLOX_PPU_JSON_URL, YOLOX_PPU_JSON)
download_file(YOLOX_NON_PPU_DXNN_URL, YOLOX_NON_PPU_DXNN)

### 5.2 Inspect the ONNX and Type 1 mapping

For YOLOX, every scale maps three named nodes:

```text
feature map ─┬─ bbox branch ─────→ bbox
             ├─ object branch ───→ obj_conf
             └─ class branch ────→ cls_conf
```

The three entries correspond to the 80×80, 40×40, and 20×20 detection scales.

In [ ]:
yolox_model = onnx.load(YOLOX_ONNX)
onnx.checker.check_model(yolox_model)
yolox_config = json.loads(YOLOX_PPU_JSON.read_text())

print("IR/opset:", yolox_model.ir_version, [(item.domain or "ai.onnx", item.version) for item in yolox_model.opset_import])
print("Inputs:", [(value.name, tensor_shape(value)) for value in yolox_model.graph.input])
print("Outputs:", [(value.name, tensor_shape(value)) for value in yolox_model.graph.output])
print(json.dumps(yolox_config["ppu"], indent=2))

node_names = {node.name for node in yolox_model.graph.node}
type1_layers = yolox_config["ppu"]["layer"]
mapped_type1_nodes = {name for layer in type1_layers for name in layer.values()}
missing_type1_nodes = sorted(mapped_type1_nodes - node_names)
if missing_type1_nodes:
    raise ValueError(f"Type 1 nodes were not found in this ONNX: {missing_type1_nodes}")
print("Verified Type 1 nodes:", sorted(mapped_type1_nodes))

The colors show the `bbox`, `obj_conf`, and `cls_conf` branches that must be paired at each scale.

![YOLOX Type 1 PPU head mapping](assets/yolox-class-n80-ppu.png)

### 5.3 Adapt the calibration-dataset path

Keep the downloaded Type 1 mapping and YOLOX preprocessing. Replace only the Model Zoo build-machine dataset path for this exercise.

In [ ]:
yolox_local_config = json.loads(YOLOX_PPU_JSON.read_text())
yolox_local_config["default_loader"]["dataset_path"] = "./calibration_dataset"
YOLOX_LOCAL_JSON = CONFIG_DIR / "yolox-s_640x640_ppu.local.json"
YOLOX_LOCAL_JSON.write_text(json.dumps(yolox_local_config, indent=2) + "\n")

print("Local config:", YOLOX_LOCAL_JSON)
print("Dataset     :", yolox_local_config["default_loader"]["dataset_path"])
print("PPU type    :", yolox_local_config["ppu"]["type"])

### 5.4 Compile and inspect the Type 1 DXNN

The next code cell is equivalent to entering the following commands in a separate terminal:

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/T05-DX-Compiler
dxcom -m models/yolox-s_640x640.onnx \
      -c configs/yolox-s_640x640_ppu.local.json \
      -o outputs/yolox_s_type1_ppu \
      --gen_log \
      --export_html
```

The code cell resolves the actual paths automatically. If the output directory already contains a DXNN file, it skips `dxcom`. Compare its `dxparse -v` output with the Type 0 result, especially the output tensor layout and PPU metadata.

In [ ]:
YOLOX_PPU_OUTPUT = OUTPUT_DIR / "yolox_s_type1_ppu"

yolox_dxnn_files = list(YOLOX_PPU_OUTPUT.glob("*.dxnn"))
if yolox_dxnn_files:
    print(f"Skip compilation: found {yolox_dxnn_files[0]}")
else:
    !source "{DX_COMPILER_VENV}/bin/activate" && \
      dxcom -m "{YOLOX_ONNX}" \
            -c "{YOLOX_LOCAL_JSON}" \
            -o "{YOLOX_PPU_OUTPUT}" \
            --gen_log \
            --export_html

yolox_dxnn_files = list(YOLOX_PPU_OUTPUT.glob("*.dxnn"))
if not yolox_dxnn_files:
    raise FileNotFoundError("Type 1 compilation did not produce a DXNN file.")
YOLOX_PPU_DXNN = max(yolox_dxnn_files, key=lambda path: path.stat().st_mtime_ns)
print("DXNN:", YOLOX_PPU_DXNN)

!dxparse -m "{YOLOX_PPU_DXNN}" -v

### 5.5 Compare PPU and non-PPU with `dxrun`

Repeat the same controlled test for YOLOX-S. The non-PPU reference is the published Model Zoo DXNN downloaded in Section 5.1; it is not compiled in this tutorial.

```bash
dxrun -m models/yolox-s_640x640_non_ppu.dxnn --use-ort -t 5
dxrun -m outputs/yolox_s_type1_ppu/<compiled-model>.dxnn --use-ort -t 5
```

Interpret this synthetic benchmark together with the task structure shown by `dxparse`. A PPU model can reduce host-side processing and output traffic even when this single FPS number does not increase.

In [ ]:
yolox_ppu_comparison = compare_ppu_runtime(
    YOLOX_NON_PPU_DXNN,
    YOLOX_PPU_DXNN,
    "Type 1 PPU DXNN",
)

In [ ]:
def run_checked(command, cwd=WORK_DIR, capture_output=False):
    command = [str(item) for item in command]
    print("$", shlex.join(command))
    options = {"cwd": cwd, "check": True, "text": True}
    if capture_output:
        options.update(stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return subprocess.run(command, **options)

## 6. YOLO26n TopK-based optimization

YOLO26 uses a one-to-one decoupled detection head. The `yolo26_postprocess` transform moves TopK selection before expensive CPU-side decoding work. This is different from PPU Type 0/1: it rewrites the ONNX graph and reduces the candidate workload handled after NPU inference.

```text
Baseline: 8,400 candidates → decode and score all candidates → TopK 300
Optimized: 8,400 candidates → TopK 300 → decode and score only 300 candidates
```

Both models keep the same detection result contract, `[1, 300, 6]`, but the order of CPU-side work changes.

### 6.1 Download and validate the baseline model

The baseline ONNX and its matching Q-Lite JSON come from the Model Zoo. The safe downloader prevents the empty-file error that can occur when a shell download fails but the next notebook cell continues.

In [ ]:
YOLO26_ONNX_URL = "https://sdk.deepx.ai/modelzoo/onnx/yolo26-n_640x640.onnx"
YOLO26_JSON_URL = "https://sdk.deepx.ai/modelzoo/q-lite-json/2_4_0/yolo26-n_640x640.json"
YOLO26_BASELINE = MODEL_DIR / "yolo26-n_640x640.onnx"
YOLO26_MODELZOO_JSON = CONFIG_DIR / "yolo26-n_640x640.modelzoo.json"

download_file(YOLO26_ONNX_URL, YOLO26_BASELINE)
download_file(YOLO26_JSON_URL, YOLO26_MODELZOO_JSON)

In [ ]:
yolo26 = onnx.load(YOLO26_BASELINE)
onnx.checker.check_model(yolo26)

print("IR/opset:", yolo26.ir_version, [(item.domain or "ai.onnx", item.version) for item in yolo26.opset_import])
print("Inputs:", [(value.name, tensor_shape(value)) for value in yolo26.graph.input])
print("Outputs:", [(value.name, tensor_shape(value)) for value in yolo26.graph.output])
print("Nodes before optimization:", len(yolo26.graph.node))

### 6.2 Verify the six head tensors

The transform needs one bounding-box tensor and one class-confidence tensor at each of the three detection scales. These are **output tensor names**, not display labels copied from another model.

In [ ]:
yolo26_head_tensors = [
    "/model.23/one2one_cv2.0/one2one_cv2.0.2/Conv_output_0",
    "/model.23/one2one_cv3.0/one2one_cv3.0.2/Conv_output_0",
    "/model.23/one2one_cv2.1/one2one_cv2.1.2/Conv_output_0",
    "/model.23/one2one_cv3.1/one2one_cv3.1.2/Conv_output_0",
    "/model.23/one2one_cv2.2/one2one_cv2.2.2/Conv_output_0",
    "/model.23/one2one_cv3.2/one2one_cv3.2.2/Conv_output_0",
]

graph_tensor_names = {name for node in yolo26.graph.node for name in node.output}
missing = [name for name in yolo26_head_tensors if name not in graph_tensor_names]
if missing:
    raise ValueError(f"Head tensors were not found: {missing}")
print("Verified all six YOLO26 head tensors.")

### 6.3 Apply `yolo26_postprocess`

The transform runs with the DX-COM Python environment because `dx_com` is installed there. It creates a separate ONNX file, so the baseline remains unchanged.

In [ ]:
%%writefile optimize_yolo26.py
from pathlib import Path
import onnx
import dx_com

work_dir = Path.cwd()
source = work_dir / "models" / "yolo26-n_640x640.onnx"
destination = work_dir / "models" / "yolo26-n_640x640_topk300.onnx"

model = onnx.load(source)
optimized = dx_com.pre_optimize(model, passes={
    "yolo26_postprocess": {
        "layers": [
            {
                "bbox": "/model.23/one2one_cv2.0/one2one_cv2.0.2/Conv_output_0",
                "cls_conf": "/model.23/one2one_cv3.0/one2one_cv3.0.2/Conv_output_0",
            },
            {
                "bbox": "/model.23/one2one_cv2.1/one2one_cv2.1.2/Conv_output_0",
                "cls_conf": "/model.23/one2one_cv3.1/one2one_cv3.1.2/Conv_output_0",
            },
            {
                "bbox": "/model.23/one2one_cv2.2/one2one_cv2.2.2/Conv_output_0",
                "cls_conf": "/model.23/one2one_cv3.2/one2one_cv3.2.2/Conv_output_0",
            },
        ],
        "num_classes": 80,
        "topk": 300,
        "input_height": 640,
        "input_width": 640,
    }
})

onnx.checker.check_model(optimized)
onnx.save(optimized, destination)
print(destination)

In [ ]:
TOPK_SCRIPT = WORK_DIR / "optimize_yolo26.py"
run_checked([DXCOM_PYTHON, TOPK_SCRIPT])

In [ ]:
YOLO26_TOPK = MODEL_DIR / "yolo26-n_640x640_topk300.onnx"
optimized_yolo26 = onnx.load(YOLO26_TOPK)
onnx.checker.check_model(optimized_yolo26)

print("Baseline outputs :", [(value.name, tensor_shape(value)) for value in yolo26.graph.output])
print("Optimized outputs:", [(value.name, tensor_shape(value)) for value in optimized_yolo26.graph.output])
print("Baseline nodes   :", len(yolo26.graph.node))
print("Optimized nodes  :", len(optimized_yolo26.graph.node))

### 6.4 Adapt the Model Zoo configuration

The same calibration and preprocessing configuration must be used for the baseline and optimized models. Only the unavailable dataset path is changed. This keeps the performance comparison controlled.

In [ ]:
yolo26_config = json.loads(YOLO26_MODELZOO_JSON.read_text())
yolo26_config["default_loader"]["dataset_path"] = "./calibration_dataset"
yolo26_local_json = CONFIG_DIR / "yolo26-n_640x640.local.json"
yolo26_local_json.write_text(json.dumps(yolo26_config, indent=2) + "\n")
print(yolo26_local_json.read_text())

### 6.5 Compile and inspect both models

Compile each ONNX into a separate output directory. Both commands use the same JSON and compiler options; the ONNX graph is the only experimental variable.

#### 6.5.1 Compile the baseline

The next code cell is equivalent to entering the following commands in a separate terminal:

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/T05-DX-Compiler
dxcom -m models/yolo26-n_640x640.onnx \
      -c configs/yolo26-n_640x640.local.json \
      -o outputs/yolo26n_baseline \
      --gen_log \
      --export_html
```

If the output directory already contains a DXNN file, the code cell reports that file and skips `dxcom`.

In [ ]:
yolo26_baseline_output = OUTPUT_DIR / "yolo26n_baseline"

baseline_dxnn_files = list(yolo26_baseline_output.glob("*.dxnn"))
if baseline_dxnn_files:
    print(f"Skip compilation: found {baseline_dxnn_files[0]}")
else:
    !source "{DX_COMPILER_VENV}/bin/activate" && \
      dxcom -m "{YOLO26_BASELINE}" \
            -c "{yolo26_local_json}" \
            -o "{yolo26_baseline_output}" \
            --gen_log \
            --export_html

baseline_dxnn_files = list(yolo26_baseline_output.glob("*.dxnn"))
if not baseline_dxnn_files:
    raise FileNotFoundError("Baseline compilation did not produce a DXNN file.")
yolo26_baseline_dxnn = max(baseline_dxnn_files, key=lambda path: path.stat().st_mtime_ns)
print("Baseline DXNN:", yolo26_baseline_dxnn)

#### 6.5.2 Compile the TopK model

The next code cell is equivalent to entering the following commands in a separate terminal:

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/T05-DX-Compiler
dxcom -m models/yolo26-n_640x640_topk300.onnx \
      -c configs/yolo26-n_640x640.local.json \
      -o outputs/yolo26n_topk300 \
      --gen_log \
      --export_html
```

If the output directory already contains a DXNN file, the code cell reports that file and skips `dxcom`.

In [ ]:
yolo26_topk_output = OUTPUT_DIR / "yolo26n_topk300"

topk_dxnn_files = list(yolo26_topk_output.glob("*.dxnn"))
if topk_dxnn_files:
    print(f"Skip compilation: found {topk_dxnn_files[0]}")
else:
    !source "{DX_COMPILER_VENV}/bin/activate" && \
      dxcom -m "{YOLO26_TOPK}" \
            -c "{yolo26_local_json}" \
            -o "{yolo26_topk_output}" \
            --gen_log \
            --export_html

topk_dxnn_files = list(yolo26_topk_output.glob("*.dxnn"))
if not topk_dxnn_files:
    raise FileNotFoundError("TopK compilation did not produce a DXNN file.")
yolo26_topk_dxnn = max(topk_dxnn_files, key=lambda path: path.stat().st_mtime_ns)
print("TopK DXNN:", yolo26_topk_dxnn)

In [ ]:
print("===== Baseline =====")
!dxparse -m "{yolo26_baseline_dxnn}" -v

print("===== TopK 300 =====")
!dxparse -m "{yolo26_topk_dxnn}" -v

### 6.6 Compare baseline and TopK performance with `dxrun`

`dxrun --use-ort -t 5` benchmarks each DXNN for five seconds with synthetic input. `--use-ort` executes the CPU subgraph as well as the NPU graph, which is required for this comparison.

Run both models on the same device under similar thermal and system-load conditions. The result measures runtime throughput, not detection accuracy or full video-pipeline FPS. Repeat the measurement for release decisions.

```bash
dxrun -m outputs/yolo26n_baseline/yolo26-n_640x640.dxnn --use-ort -t 5
dxrun -m outputs/yolo26n_topk300/lo26-n_640x640_topk300.dxnn --use-ort -t 5
```

In [ ]:
from IPython.display import Markdown, display

def benchmark_dxnn(label, model_path, seconds=5):
    result = run_checked(
        [DXRUN_PATH, "-m", model_path, "--use-ort", "-t", str(seconds)],
        capture_output=True,
    )
    print(result.stdout)
    match = re.search(r"FPS\s*:\s*([0-9.]+)", result.stdout)
    if not match:
        raise RuntimeError(f"Could not read FPS from dxrun output for {label}.")
    return float(match.group(1))

baseline_fps = benchmark_dxnn("Baseline", yolo26_baseline_dxnn)
topk_fps = benchmark_dxnn("TopK 300", yolo26_topk_dxnn)
speedup = topk_fps / baseline_fps

comparison = (
    "| Model | FPS | Relative throughput |\n"
    "|---|---:|---:|\n"
    f"| Baseline | {baseline_fps:.2f} | 1.00× |\n"
    f"| TopK 300 | {topk_fps:.2f} | {speedup:.2f}× |"
)
display(Markdown(comparison))

## Summary

In this tutorial, you learned how to:

- treat calibration data and preprocessing as part of the model definition,
- separate hardware PPU processing from CPU-side TopK graph optimization,
- identify the boundary between PPU filtering/class selection and host-CPU NMS,
- choose Type 0 for an anchor-based YOLOv7 head and verify its Conv-node mapping,
- choose Type 1 for an anchor-free YOLOX head and verify its bbox/object/class branches,
- download Model Zoo artifacts safely and adapt only environment-specific configuration values,
- compile and inspect real Type 0 and Type 1 PPU DXNN models,
- compare PPU and non-PPU Model Zoo DXNN throughput with `dxrun --use-ort`,
- explain why reduced host work can be valuable even when PPU does not increase synthetic-input FPS,
- apply TopK-first optimization to YOLO26n while preserving the output contract,
- compare baseline and TopK DXNN throughput with `dxrun --use-ort`.

Continue with the Advanced tutorial for programmatic compiler use, diagnosis, Q-PRO, QXNN resume, QAT, and advanced compiler controls.